### Célula 1 — Carregando a dim_ticket_description da Gold

In [0]:
import pyspark.sql.functions as F

GOLD_PATH = "/Volumes/workspace/default/raw/gold/"

# Lendo a dimensão de descrição
df_desc = spark.read.format("delta").load(f"{GOLD_PATH}dim_ticket_description/")

print(f"Total de registros: {df_desc.count():,}")
print(f"Colunas: {df_desc.columns}")
print()
df_desc.show(3, truncate=80)

Total de registros: 8,469
Colunas: ['Ticket_ID', 'Ticket_Description_Original', 'Description_Type', 'Description_Clean']

+---------+--------------------------------------------------------------------------------+------------------+--------------------------------------------------------------------------------+
|Ticket_ID|                                                     Ticket_Description_Original|  Description_Type|                                                               Description_Clean|
+---------+--------------------------------------------------------------------------------+------------------+--------------------------------------------------------------------------------+
|       12|I'm having an issue with the [produto]. Please assist.\n\n4. It is possible t...|Issue with Product|\n\n4. It is possible that we cannot find some type of text or a product name...|
|       18|I'm having an issue with the [produto]. Please assist. Thanks!"\n\n* [0] - [0...|Issue with Pro

### Célula 2 — Análise do que precisa ser limpo

In [0]:
import builtins

print("=" * 60)
print("DIAGNÓSTICO — RUÍDOS NO DESCRIPTION_CLEAN")
print("=" * 60)

total = df_desc.count()

# Quebras de linha
quebras = df_desc.filter(
    F.col("Description_Clean").contains("\n")
).count()
print(f"Com quebras de linha (\\n):     {quebras:,} ({builtins.round(quebras/total*100,1)}%)")

# Placeholder [produto] ainda presente
placeholder = df_desc.filter(
    F.col("Description_Clean").contains("[produto]")
).count()
print(f"Com placeholder [produto]:     {placeholder:,} ({builtins.round(placeholder/total*100,1)}%)")

# Padrão [0] - [0]
padrao_zero = df_desc.filter(
    F.col("Description_Clean").contains("[0]")
).count()
print(f"Com padrão [0]:                {padrao_zero:,} ({builtins.round(padrao_zero/total*100,1)}%)")

# Textos muito curtos (menos de 20 caracteres)
muito_curtos = df_desc.filter(
    F.length(F.col("Description_Clean")) < 20
).count()
print(f"Textos muito curtos (<20 chars): {muito_curtos:,}")

# Comprimento médio
avg_len = df_desc.agg(
    F.round(F.avg(F.length("Description_Clean")), 0).alias("media"),
    F.min(F.length("Description_Clean")).alias("minimo"),
    F.max(F.length("Description_Clean")).alias("maximo")
).collect()[0]
print(f"\nComprimento médio:  {int(avg_len['media'])} chars")
print(f"Comprimento mínimo: {avg_len['minimo']} chars")
print(f"Comprimento máximo: {avg_len['maximo']} chars")

# Distribuição por Description_Type
print()
print("─" * 60)
print("Distribuição por Description_Type:")
df_desc.groupBy("Description_Type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

DIAGNÓSTICO — RUÍDOS NO DESCRIPTION_CLEAN
Com quebras de linha (\n):     6,150 (72.6%)
Com placeholder [produto]:     5,947 (70.2%)
Com padrão [0]:                6 (0.1%)
Textos muito curtos (<20 chars): 0

Comprimento médio:  235 chars
Comprimento mínimo: 86 chars
Comprimento máximo: 377 chars

────────────────────────────────────────────────────────────
Distribuição por Description_Type:
+--------------------+-----+
|    Description_Type|count|
+--------------------+-----+
|  Issue with Product| 5868|
|      Account Access|  189|
|      Password Reset|  186|
|  Network Connection|  184|
|        Software Bug|  183|
|      Hardware Noise|  182|
|       Network Setup|  180|
|     General Problem|  179|
|    Software Bug App|  179|
|     WiFi Connection|  178|
|    Hardware Problem|  169|
|Data Loss Accidental|  165|
|     Software Glitch|  164|
|     Data Loss Issue|  164|
|       Login Problem|  154|
|        Device Crash|  145|
+--------------------+-----+



#### Diagnóstico — Ruídos no Description_Clean

| Problema | Volume | % | Prioridade |
|---------|--------|---|-----------|
| Quebras de linha `\n` | 6.150 | 72.6% | 🔴 Alta |
| Placeholder `[produto]` | 5.947 | 70.2% | 🔴 Alta |
| Padrão `[0]` | 6 | 0.1% | 🟡 Baixa |
| Textos muito curtos | 0 | 0% | ✅ OK |

#### Comprimento do texto
| Métrica | Valor |
|---------|-------|
| Média | 235 chars |
| Mínimo | 86 chars |
| Máximo | 377 chars |

#### Distribuição por Tipo
- **Issue with Product domina com 5.868 registros (69.3%)** — categoria muito ampla
- 15 categorias específicas bem distribuídas entre 145 e 189 registros
- Sem registros "Outro" — classificação 100% coberta ✅

#### Conclusão
O campo `Description_Clean` ainda contém ruídos significativos.
Será necessário criar o campo `Description_NLP` com limpeza
profunda para uso em modelos de linguagem e análise de sentimento.

### Célula 3 — Limpeza profunda para NLP

In [0]:
from pyspark.sql.functions import regexp_replace, trim, lower       # funções de texto

df_nlp = df_desc.withColumn(
    "Description_NLP",

    # 1. Remove quebras de linha e tabs
    trim(regexp_replace(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        regexp_replace(
                            regexp_replace(
                                F.col("Description_Clean"),
                                r"\n",  " "                          # quebra de linha → espaço
                            ),
                            r"\t", " "                               # tab → espaço
                        ),
                        r"\r", " "                                   # carriage return → espaço
                    ),
                    r"\[produto\]", ""                               # remove [produto]
                ),
                r"\[0\](\s*-\s*\[0\])*", ""                        # remove padrão [0]-[0]
            ),
            r"Thanks[!.]?\s*\"?", ""                                 # remove "Thanks!"
        ),
        r"\s{2,}", " "                                               # múltiplos espaços → um
    ))
)

# Validando
print("=" * 60)
print("VALIDAÇÃO — Description_NLP")
print("=" * 60)

total = df_nlp.count()

quebras = df_nlp.filter(F.col("Description_NLP").contains("\n")).count()
placeholder = df_nlp.filter(F.col("Description_NLP").contains("[produto]")).count()
padrao_zero = df_nlp.filter(F.col("Description_NLP").contains("[0]")).count()

print(f"Quebras de linha restantes:  {quebras:,} {'✅' if quebras == 0 else '🔴'}")
print(f"Placeholder restante:        {placeholder:,} {'✅' if placeholder == 0 else '🔴'}")
print(f"Padrão [0] restante:         {padrao_zero:,} {'✅' if padrao_zero == 0 else '🔴'}")

# Comprimento após limpeza
avg_len = df_nlp.agg(
    F.round(F.avg(F.length("Description_NLP")), 0).alias("media"),
    F.min(F.length("Description_NLP")).alias("minimo"),
    F.max(F.length("Description_NLP")).alias("maximo")
).collect()[0]

print(f"\nComprimento médio:  {int(avg_len['media'])} chars")
print(f"Comprimento mínimo: {avg_len['minimo']} chars")
print(f"Comprimento máximo: {avg_len['maximo']} chars")

print()
df_nlp.select("Ticket_ID", "Description_NLP").show(3, truncate=80)

VALIDAÇÃO — Description_NLP
Quebras de linha restantes:  0 ✅
Placeholder restante:        0 ✅
Padrão [0] restante:         0 ✅

Comprimento médio:  224 chars
Comprimento mínimo: 75 chars
Comprimento máximo: 358 chars

+---------+--------------------------------------------------------------------------------+
|Ticket_ID|                                                                 Description_NLP|
+---------+--------------------------------------------------------------------------------+
|       12|4. It is possible that we cannot find some type of text or a product name to ...|
|       18|* If this product is sold and you have not used any I've reviewed the trouble...|
|       38|I've forgotten my password for my account, and the password reset option is n...|
+---------+--------------------------------------------------------------------------------+
only showing top 3 rows


### Célula 4 — Análise de sentimento com TextBlob

In [0]:
import subprocess
subprocess.run(["pip", "install", "textblob", "--quiet"])            # instala TextBlob
from textblob import TextBlob                                        # biblioteca NLP
import pandas as pd

# Convertendo para Pandas — volume gerenciável
df_nlp_pd = df_nlp.select(
    "Ticket_ID",
    "Description_Type",
    "Description_NLP"
).toPandas()

# Calculando polaridade e subjetividade
def get_sentiment(texto):
    if not texto or len(texto.strip()) == 0:
        return 0.0, 0.0
    blob = TextBlob(str(texto))
    return builtins.round(blob.sentiment.polarity, 3), \
           builtins.round(blob.sentiment.subjectivity, 3)

print("Calculando sentimento — aguarde...")

df_nlp_pd[["polarity", "subjectivity"]] = df_nlp_pd["Description_NLP"].apply(
    lambda x: pd.Series(get_sentiment(x))                           # aplica em cada linha
)

# Classificando o sentimento
def classify_sentiment(polarity):
    if polarity > 0.1:   return "Positivo"
    if polarity < -0.1:  return "Negativo"
    return "Neutro"

df_nlp_pd["Sentiment"] = df_nlp_pd["polarity"].apply(classify_sentiment)

print(f"✅ Sentimento calculado para {len(df_nlp_pd):,} registros!")
print()
print("Distribuição de sentimento:")
print(df_nlp_pd["Sentiment"].value_counts())
print()
print("Polaridade média por tipo:")
print(df_nlp_pd.groupby("Description_Type")["polarity"].mean().round(3).sort_values())

Calculando sentimento — aguarde...
✅ Sentimento calculado para 8,469 registros!

Distribuição de sentimento:
Sentiment
Neutro      4302
Positivo    2868
Negativo    1299
Name: count, dtype: int64

Polaridade média por tipo:
Description_Type
Account Access         -0.238
Network Connection     -0.180
Hardware Problem       -0.178
Software Glitch        -0.076
Hardware Noise          0.006
Network Setup           0.007
WiFi Connection         0.044
Device Crash            0.051
Password Reset          0.051
Login Problem           0.066
Data Loss Issue         0.067
Issue with Product      0.075
Software Bug App        0.084
Software Bug            0.135
Data Loss Accidental    0.206
General Problem         0.215
Name: polarity, dtype: float64


#### Análise de Sentimento — Description_NLP

### Distribuição Geral
| Sentimento | Quantidade | % |
|-----------|------------|---|
| Neutro | 4.302 | 50.8% |
| Positivo | 2.868 | 33.9% |
| Negativo | 1.299 | 15.3% |

#### Polaridade por Tipo — Do mais negativo ao mais positivo
| Tipo | Polaridade | Sentimento |
|------|-----------|-----------|
| Account Access | -0.238 | 🔴 Mais negativo |
| Network Connection | -0.180 | 🔴 Negativo |
| Hardware Problem | -0.178 | 🔴 Negativo |
| Software Glitch | -0.076 | 🟡 Levemente negativo |
| Hardware Noise | +0.006 | ⚪ Neutro |
| Network Setup | +0.007 | ⚪ Neutro |
| WiFi Connection | +0.044 | 🟡 Levemente positivo |
| Device Crash | +0.051 | 🟡 Levemente positivo |
| Password Reset | +0.051 | 🟡 Levemente positivo |
| Login Problem | +0.066 | 🟡 Levemente positivo |
| Data Loss Issue | +0.067 | 🟡 Levemente positivo |
| Issue with Product | +0.075 | 🟢 Positivo |
| Software Bug App | +0.084 | 🟢 Positivo |
| Software Bug | +0.135 | 🟢 Positivo |
| Data Loss Accidental | +0.206 | 🟢 Positivo |
| General Problem | +0.215 | 🟢 Mais positivo |

#### Insights
- **Account Access é o tipo mais negativo (-0.238)** — clientes frustrados ao perder acesso
- **Network Connection e Hardware Problem** também muito negativos — problemas físicos geram mais frustração
- **General Problem e Data Loss Accidental** surpreendentemente positivos — clientes descritivos e calmos
- **50.8% neutros** — linguagem técnica e objetiva domina as descrições
- Correlação esperada: tipos com menor satisfação têm maior negatividade no texto

### Célula 5 — Salvando o campo NLP na Gold

In [0]:
# Convertendo de volta para Spark com os novos campos
df_nlp_spark = spark.createDataFrame(df_nlp_pd)                     # Pandas → Spark

# Salvando na Gold — atualizando dim_ticket_description
(df_nlp_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{GOLD_PATH}dim_ticket_description/")
)

# Validando
df_check = spark.read.format("delta").load(f"{GOLD_PATH}dim_ticket_description/")
print(f"Linhas: {df_check.count():,}")
print(f"Colunas: {df_check.columns}")
print()

# Distribuição final
df_check.groupBy("Sentiment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

print("✅ dim_ticket_description atualizada com NLP!")

Linhas: 8,469
Colunas: ['Ticket_ID', 'Description_Type', 'Description_NLP', 'polarity', 'subjectivity', 'Sentiment']

+---------+-----+
|Sentiment|count|
+---------+-----+
|   Neutro| 4302|
| Positivo| 2868|
| Negativo| 1299|
+---------+-----+

✅ dim_ticket_description atualizada com NLP!
